RAG 프로세스는 인덱싱 과정과 쿼리 과정 두가지로 나뉜다. 
## 인덱싱 과정 
문서를 수집하고 검색이 용이하도록 데이터베이스에 적재한다. 

#### **문서 준비 및 분할**
- 다양한 문서 수집 후 작은 조각으로 분할한다. 
- 이 청크 들은 의미 단위로 나누어지며 이때 적절한 크기로 나누는 것이 중요하다. 
- 너무 작으면 정보가 적고 너무 크면 검색 시에 정확도가 떨어진다. 
#### 임베딩 생성 및 저장 
- 청크로 나뉜 문서를 임베딩 모델을 사용해 벡터 형태로 변환한다. 
- 임베딩은 청크의 의미를 숫자로 표현한 벡터로, 벡터 데이터베이스에 저장되어 나중에 관련 정보를 빠르게 찾는데 사용된다. 
#### 데이터베이스 적재 및 관리 
- 변환된 임베딩 벡터는 벡터 데이터베이스에 저장되며 이를 통해 검색의 효율을 높인다. 


## 쿼리 과정 
사용자가 질문을 입력하면 관련정보를 실시간으로 검색하고 답변을 생성하는 단계이다.
####  질문 입력 및 변환 
- 사용자가 질문을 입력하면 시스템은 해당 질문을 데이터베이스 검색에 사용할 수 있도록 벡터롤 변환 하며 이전 대화 맥락에 반영해서 재작성 가능하다. 
#### 검색 및 재정렬 
- 변환된 질문 벡터는 데이터베이스에서 연관성이 높은 문서 청크들을 검색하는데 사용된다. 
- 검색된 청크 들은 유사도 점수에 따라 상위 k개의 청크가 선택된다. 
#### 프롬프트 템플릿 설정 
- 사용자 질문에 적절히 답변을 생성하도록 템플릿을 설정한다. 
- 프롬프트 템플릿은 답변 생성과정에서 입력된 질문과 문서 청크를 결합해서 대화형 응답을 만든다. 
#### 문맥 구성 
- 검색된 청크들을 재조합해서 질문에 맞는 문맥을 형성한다. 
- 생성 모데이 가진 토큰 수를 제한을 고려하여 필요한 부분만 포함하여 최적의 답변을 생성하도록 조정한다. 
#### 답변 및 응답제공
- 최종적으로 구성된 문맥과 질문을 함께 모델에 입력해서 답변을 생성한다.  

In [2]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_openai import OpenAI, OpenAIEmbeddings 
from langchain_chroma import Chroma 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


#### 인덱스 과정 1- 문서 준비 및 분할 

In [3]:
loader = PyPDFLoader("./data/2024_KB_부동산_보고서_최종.pdf")
documents = loader.load()
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_spliter.split_documents(documents)

print(f"분할된 청크의 수 {len(chunks)}")


분할된 청크의 수 135


In [4]:
import chromadb

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    tenant="default_tenant",
    database="default_database",
)

print(client.heartbeat())

1782386707033202050


In [ ]:
embedding_function = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

In [6]:
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding_function,
    collection_name="real_estate",
    client = client
)

In [7]:
retriever = vector_db.as_retriever(search_kwargs={"k":3})

In [9]:
from langchain_openai import ChatOpenAI
templeate = """
    당신은 KB 부동산 보고서 전문가입니다.\n
    다음 정보를 바탕으로 사용자의 질문에 답변해주세요.\n
    컨텍스트 : {context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", templeate),
        ("placeholder", "{chat_history}"),
        ("human", "{question}")
    ]
)

model = ChatOpenAI(
    model="gemma-3-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

In [10]:
print(prompt.format(context="컨텍스트 예시", chat_hisotry=["대화 기록 예시1", "대화 기록 예시2"], question="질문 예시"))

System: 
    당신은 KB 부동산 보고서 전문가입니다.

    다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

    컨텍스트 : 컨텍스트 예시

Human: 질문 예시


In [11]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | model
)

In [ ]:
chat_history = ChatMessageHistory()

chain_with_memory = RunnableWithMessageHistory(
    chain, 
    lambda session_id : chat_history,
    input_messages_key="question", 
    history_messages_key="chat_history"
)

In [ ]:

def chat_with_bot():
    session_id = "user_session"
    print("KB 부동산 보고서 챗봇입니다. 질문해 주세요. (종료하려면 'quit' 입력)")
    while True:
        user_input = input("사용자: ")
        if user_input.lower() == 'quit':
            break

        response = chain_with_memory.invoke(
            {"question": user_input},
            {"configurable": {"session_id": session_id}}
        )

        print("챗봇:", response)


if __name__ == "__main__":
    chat_with_bot()